In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score

In [17]:
df=pd.read_csv("course_lead_scoring_2026.csv")

In [18]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [19]:
print(df.isna().sum())

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64


In [20]:
cat_cols = df.select_dtypes(include=['object', 'category']).columns
num_cols = df.select_dtypes(include=['number']).columns

In [21]:
df[cat_cols] = df[cat_cols].fillna('NA')
df[num_cols] = df[num_cols].fillna(0.0)

In [22]:
print(df.isna().sum())


lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64


In [23]:
df['industry'].mode()[0]

'technology'

In [24]:
numerical_features = ['annual_income','number_of_courses_viewed','interaction_count','lead_score']
print(df[numerical_features].corr())


                          annual_income  number_of_courses_viewed  \
annual_income                  1.000000                  0.161300   
number_of_courses_viewed       0.161300                  1.000000   
interaction_count              0.122842                  0.721609   
lead_score                     0.229496                  0.757204   

                          interaction_count  lead_score  
annual_income                      0.122842    0.229496  
number_of_courses_viewed           0.721609    0.757204  
interaction_count                  1.000000    0.915746  
lead_score                         0.915746    1.000000  


In [25]:
y = df['converted'].values
df_features = df.drop(columns=['converted'])


In [26]:
df_full_train, df_test = train_test_split(df, test_size = 0.2, random_state = 42)
df_train, df_val = train_test_split(df_full_train, test_size = 0.25, random_state = 42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [27]:
categorical = ['industry', 'location', 'lead_source', 'employment_status']

for col in categorical:
    score = mutual_info_score(df_train[col], y_train)
    print(f"{col}: {round(score, 2)}")

industry: 0.0
location: 0.0
lead_source: 0.03
employment_status: 0.02


In [28]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

categorical = ['lead_source', 'industry', 'employment_status', 'location']
numerical = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
val_dicts = df_val[categorical + numerical].to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
accuracy = accuracy_score(y_val, y_pred)

print(f"Validation Accuracy: {round(accuracy, 2)}")

Validation Accuracy: 0.65


In [29]:
all_features = [
    'lead_source', 'industry', 'employment_status', 'location',
    'annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score'
]

# Baseline accuracy
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(df_train[all_features].to_dict(orient='records'))
X_val = dv.transform(df_val[all_features].to_dict(orient='records'))

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)
orig_acc = accuracy_score(y_val, model.predict(X_val))

# Feature elimination for target options
options = ['lead_source', 'number_of_courses_viewed', 'interaction_count']

for feat in options:
    features_sub = [f for f in all_features if f != feat]

    dv_sub = DictVectorizer(sparse=False)
    X_train_sub = dv_sub.fit_transform(df_train[features_sub].to_dict(orient='records'))
    X_val_sub = dv_sub.transform(df_val[features_sub].to_dict(orient='records'))

    m = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    m.fit(X_train_sub, y_train)

    acc_sub = accuracy_score(y_val, m.predict(X_val_sub))
    diff = orig_acc - acc_sub
    print(f"{feat}: diff = {diff:.6f}")

lead_source: diff = 0.003000
number_of_courses_viewed: diff = 0.002000
interaction_count: diff = 0.044000


In [30]:

C_values = [0.000001, 0.00001, 0.0001, 0.001]

for c in C_values:
    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    acc = accuracy_score(y_val, y_pred)

    print(f"C={c}: accuracy = {round(acc, 3)}")

C=1e-06: accuracy = 0.598
C=1e-05: accuracy = 0.598
C=0.0001: accuracy = 0.613
C=0.001: accuracy = 0.645
